# Graph-DTW: match → resolve → visualize

End-to-end run of the route-based pipeline. The two visualizations are generated by **running the existing scripts** (`scripts/graph_dtw_validation_map.py` and `scripts/graph_dtw_edge_detail.py`), so the saved `.html` are the **exact same standalone interactive maps** you get from the command line — the notebook just previews them inline with `IFrame`.

1. **Match** every OSM A-edge to a connected NVDB B-route (`match_routes`).
2. **Resolve** — drop low-quality matches with data-driven thresholds (`suggest_thresholds` →
   `resolve_routes`).
3. **Visualize** two ways — whole-network **validation map** and a **single edge → subgraph** detail.

In [1]:
import sys, os, subprocess
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../scripts'))   # so we can import the script loader
from IPython.display import IFrame
from network_matching import suggest_thresholds, setup_logging
from graph_dtw_edge_detail import load_matcher   # same table names both scripts use
setup_logging(console=False)

REPO = os.path.abspath('..')          # scripts expect to run from the repo root
PY = sys.executable

m = load_matcher('../data/osm_edges.csv', '../data/sweden_edges.csv', 3006, 30.0)
print('matcher ready')

matcher ready


## 1. Run map matching

In [2]:
rl, rs = m.match_routes(n_jobs=-1)
print(f'matched: {(rs.match_type != "NO_MATCH").sum()}   NO_MATCH: {(rs.match_type == "NO_MATCH").sum()}')
rs[rs.match_type != 'NO_MATCH'][['source_id','n_edges','dtw_distance','max_dtw_distance',
                                 'bearing_diff','overlap_pct','match_type']].head()

matched: 3344   NO_MATCH: 604


,source_id,n_edges,dtw_distance,max_dtw_distance,bearing_diff,overlap_pct,match_type
0,1,1,2.059110,3.764224,2.058297,100,1:1
1,2,2,2.722038,5.100140,2.112001,100,1:N_ROUTE
2,3,3,2.609373,6.259154,1.439246,100,1:N_ROUTE
3,4,1,24.904947,69.570176,14.855059,62,1:1
4,5,1,2.133325,5.818386,1.614251,100,1:1


## 2. Estimate thresholds and resolve
`suggest_thresholds` picks data-driven cuts; `resolve_routes` turns failing routes into `NO_MATCH`. (The validation-map script below re-applies these same cuts via `--auto-thresholds`.)

In [3]:
sugg = suggest_thresholds(rs, report=True)
kw = {k: v for k, v in sugg['recommended'].items() if v is not None}
rs_res, rl_res = m.resolve_routes(rs, rl, **kw)
print('\napplied:', kw)
print('after resolve -> matched:', int((rs_res.match_type != 'NO_MATCH').sum()),
      '  NO_MATCH:', int((rs_res.match_type == 'NO_MATCH').sum()))

      QUALITY THRESHOLD SUGGESTIONS (resolve_routes)
rows total             : 3948   (NO_MATCH excluded: 604)

-- dtw_distance (lower better, n=3344) --
   summary   : p50=2.69  p90=30.14  p95=45.05  p99=84.10  max=382.55
   iqr=20.47  mad=9.71  p97.5=59.72  otsu=23.92
   kde_valley=10.81  gmm=5.69 (sep=1.6)  kneedle=8.12  iforest=7.42
   -> max_match_dist = 7.418   [bimodal (KDE valley (GMM sep=1.6<2.0)): good/bad crossover]

-- max_dtw_distance (lower better, n=3344) --
   summary   : p50=5.11  p90=56.93  p95=84.25  p99=160.53  max=820.06
   iqr=42.71  mad=16.25  p97.5=114.94  otsu=49.96
   kde_valley=24.85  gmm=10.34 (sep=1.6)  kneedle=13.88  iforest=12.87
   -> max_max_dist = 12.874   [bimodal (KDE valley (GMM sep=1.6<2.0)): good/bad crossover]

-- bearing_diff (lower better, n=3344) --
   summary   : p50=1.49  p90=59.94  p95=74.74  p99=88.32  max=179.31
   iqr=28.85  mad=7.57  p97.5=82.46  otsu=32.81
   kde_valley=14.93  gmm=4.26 (sep=1.9)  kneedle=7.11  iforest=4.73
   -> max_bea

## 3. Visualization A — validation map (whole network)
Runs `scripts/graph_dtw_validation_map.py --auto-thresholds` → `output/graph_dtw_validation_map_resolved.html` (identical to the standalone CLI output). Toggle layers top-right: A matched / NO_MATCH / under-covered, B used / unused / under-/over-used.

In [4]:
subprocess.run([PY, 'scripts/graph_dtw_validation_map.py', '--auto-thresholds'],
               cwd=REPO, check=True)
IFrame('../output/graph_dtw_validation_map_resolved.html', width='100%', height=600)

2026-06-01 23:45:36,270 | INFO    | network_matching | Logging initialized -> logs/network_matching_20260601_234536.log
2026-06-01 23:45:36,365 | INFO    | network_matching.scripts.graph_dtw_validation_map | running match_routes...


2026-06-01 23:45:36,510 | INFO    | network_matching.matcher | graph-DTW: 25429 candidate pairs over 3408 A-edges (snap=0.75m, step=10.00m, n_jobs=-1)


2026-06-01 23:45:37,247 | INFO    | network_matching.matcher | graph-DTW: 3408 A-edge tasks to align
2026-06-01 23:45:37,247 | INFO    | network_matching.matcher | graph-DTW: aligning in parallel (n_jobs=-1)...


2026-06-01 23:45:41,712 | INFO    | network_matching.matcher | graph-DTW: alignment finished in 4.5s
2026-06-01 23:45:41,778 | INFO    | network_matching.matcher | graph-DTW: 3344 matched (1237 multi-edge routes), 604 NO_MATCH; 5141 route-edge rows; total 5.3s


      QUALITY THRESHOLD SUGGESTIONS (resolve_routes)
rows total             : 3948   (NO_MATCH excluded: 604)

-- dtw_distance (lower better, n=3344) --
   summary   : p50=2.69  p90=30.14  p95=45.05  p99=84.10  max=382.55
   iqr=20.47  mad=9.71  p97.5=59.72  otsu=23.92
   kde_valley=10.81  gmm=5.69 (sep=1.6)  kneedle=8.12  iforest=7.42
   -> max_match_dist = 7.418   [bimodal (KDE valley (GMM sep=1.6<2.0)): good/bad crossover]

-- max_dtw_distance (lower better, n=3344) --
   summary   : p50=5.11  p90=56.93  p95=84.25  p99=160.53  max=820.06
   iqr=42.71  mad=16.25  p97.5=114.94  otsu=49.96
   kde_valley=24.85  gmm=10.34 (sep=1.6)  kneedle=13.88  iforest=12.87
   -> max_max_dist = 12.874   [bimodal (KDE valley (GMM sep=1.6<2.0)): good/bad crossover]

-- bearing_diff (lower better, n=3344) --
   summary   : p50=1.49  p90=59.94  p95=74.74  p99=88.32  max=179.31
   iqr=28.85  mad=7.57  p97.5=82.46  otsu=32.81
   kde_valley=14.93  gmm=4.26 (sep=1.9)  kneedle=7.11  iforest=4.73
   -> max_bea

2026-06-01 23:45:44,823 | INFO    | network_matching.scripts.graph_dtw_validation_map | saved -> output/graph_dtw_validation_map_resolved.html
Saved output/graph_dtw_validation_map_resolved.html


## 4. Visualization B — single edge → subgraph
Runs `scripts/graph_dtw_edge_detail.py --edge-id <id>` → `output/graph_dtw_edge_<id>.html` (identical to the standalone CLI output): the local B-subgraph, the chosen route, and **every point match** (point-to-node vs point-to-projection), with the per-edge coverage table.

In [5]:
edge_id = 1278   # change to inspect any OSM edge_id (e.g. a high-drift one from the validation map)
subprocess.run([PY, 'scripts/graph_dtw_edge_detail.py', '--edge-id', str(edge_id)],
               cwd=REPO, check=True)
IFrame(f'../output/graph_dtw_edge_{edge_id}.html', width='100%', height=600)

2026-06-01 23:45:46,161 | INFO    | network_matching | Logging initialized -> logs/network_matching_20260601_234546.log


2026-06-01 23:45:46,385 | INFO    | network_matching.scripts.graph_dtw_edge_detail | edge 1278: 18 candidate B-edges


2026-06-01 23:45:46,838 | INFO    | network_matching.scripts.graph_dtw_edge_detail | saved -> output/graph_dtw_edge_1278.html
Saved output/graph_dtw_edge_1278.html  (route=[1794, 927], avg match dist=2.50 m)


Both maps are real standalone `.html` files under `output/` — open them directly in a browser for the full-screen interactive view. Re-run cell 4 with a different `edge_id` to inspect any edge.